# B7. MLP on Word2Vec: The Stopper

*ML and NLP course - Data Trainers LLC - Axel Sirota*

## Everything converges here

This is the notebook the whole course has been climbing toward. Look at what you now hold:

- From B5: pretrained word vectors (`wv`) and a way to turn a sentence into one fixed-size
  vector by averaging them (`doc_vector`). You also measured a LogisticRegression baseline
  on those vectors.
- From B6: how to build a model with `nn.Module`, pick a loss, pick an optimizer, batch data
  with a `DataLoader`, and run a training loop.

We snap those two halves together. Text becomes a 100-dimensional vector (pretrained word2vec,
averaged), and that vector feeds a small neural network you train to classify sentiment. This
is the **embeddings as features** pattern: freeze a pretrained encoder, train a small head on
top of its output. It is the foundation under every modern transfer-learning system, and it is
the direct precursor to fine-tuning a transformer in Part C.

The bar is concrete. Your MLP must BEAT the LogisticRegression baseline on the exact same
features. If a linear model already does well, a nonlinear one should do at least a little
better, and we will see exactly why.

## What you will be able to do

- Load (or rebuild) a 100-d averaged-word2vec feature matrix `X, y` for SST-2 sentiment.
- Fit and measure the LogisticRegression baseline: the bar to beat.
- Define and train an MLP head on those features with the B6 machinery.
- Evaluate against the baseline, read a confusion matrix, and find where the model fails.
- Explain why averaging word vectors caps accuracy, and what fixes it (attention, Part C).

## Prerequisites

- B5: `wv`, `doc_vector`, cosine geometry, the LogisticRegression baseline idea.
- B6: `nn.Module`, `nn.Linear`, `nn.ReLU`, `CrossEntropyLoss`, `Adam`, `TensorDataset`,
  `DataLoader`, the training loop.

## Session format

Theory -> Demo -> Lab, four labs. Runs on Colab CPU in a few minutes; a GPU is optional and
barely matters because the model is tiny.

## Section 0. Setup

We install pinned versions so the gensim word-vector loader works. Two pins matter:

- `numpy<2` and `scipy<1.13`: gensim 4.3 imports `scipy.linalg.triu`, which was removed in
  scipy 1.13. Without these pins you get `ImportError: cannot import name 'triu'`.
- `gensim==4.3.3`: brings the `gensim.downloader` API and the `key_to_index` vocabulary
  interface we use to guard against out-of-vocabulary words.

Colab ships numpy 2.x preinstalled, so after the install cell you may need to restart the
runtime (Runtime -> Restart runtime) once, then run the notebook from the top. This is the same
restart you did in B5.

In [ ]:
# Pinned install. Restart the Colab runtime once after this cell if prompted, then run from top.
!pip install -q "numpy<2" "scipy<1.13" "gensim==4.3.3" "datasets>=2.19,<3" scikit-learn

# Notes:
# - numpy<2 and scipy<1.13 keep gensim 4.3 importable (scipy.linalg.triu was removed in 1.13).
# - datasets<3 keeps the GLUE/SST-2 loader stable (datasets 4.x dropped script-based loaders).
# - scikit-learn gives us StandardScaler, LogisticRegression, and the metrics.
# - torch, pandas, numpy, matplotlib, seaborn are preinstalled on Colab.

In [ ]:
# Imports
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility: same SEED as B4/B5/B6
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device: a torch.device object (the B4/B5/B6 idiom)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Hyperparameters, all in one place
N_SAMPLES   = 2000   # SST-2 rows to subsample (fast on CPU); raise for a stronger model
INPUT_DIM   = 100    # averaged word2vec dimension (glove-wiki-gigaword-100). NOT 384 (SBERT).
HIDDEN_DIM  = 128
N_CLASSES   = 2      # SST-2: negative / positive
DROPOUT     = 0.3
LR          = 1e-3
BATCH_SIZE  = 64
EPOCHS_MAX  = 50
PATIENCE    = 6      # early-stopping patience (epochs without val improvement)

In [ ]:
# Get the B7 input: a 100-d averaged-word2vec feature matrix X and labels y for SST-2.
# Path 1: load the X.npy / y.npy you saved in the B5 homework (fast, no re-download).
# Path 2 (fallback): rebuild them here from pretrained word vectors + a SST-2 subsample.

import os

def load_saved_features():
    """Try to load the B5-homework features. Returns (X, y) or None."""
    if os.path.exists('X.npy') and os.path.exists('y.npy'):
        X = np.load('X.npy')
        y = np.load('y.npy')
        if X.shape[1] == INPUT_DIM:
            print(f"Loaded saved features from B5: X={X.shape}, y={y.shape}")
            return X, y
    return None

loaded = load_saved_features()

if loaded is not None:
    X, y = loaded
else:
    print("No saved features found. Rebuilding from pretrained word vectors (this is the B5 recipe).")
    import re
    import gensim.downloader as api
    from datasets import load_dataset

    # Pretrained word vectors, exactly as in B5: 400k words, 100 dimensions.
    wv = api.load('glove-wiki-gigaword-100')

    def doc_vector(text):
        """Mean-pool the in-vocab word vectors of a text into one 100-d vector (B5 recipe)."""
        tokens = re.findall(r"[a-z']+", text.lower())
        vecs = [wv[t] for t in tokens if t in wv.key_to_index]  # OOV guard
        if not vecs:
            return np.zeros(INPUT_DIM, dtype=np.float32)
        return np.mean(vecs, axis=0).astype(np.float32)

    # SST-2 sentiment: fields 'sentence' and 'label' (0=negative, 1=positive).
    ds = load_dataset('nyu-mll/glue', 'sst2', split='train')
    ds = ds.shuffle(seed=SEED).select(range(N_SAMPLES))

    X = np.vstack([doc_vector(s) for s in ds['sentence']]).astype(np.float32)
    y = np.array(ds['label'], dtype=np.int64)
    np.save('X.npy', X)  # cache so a re-run is instant
    np.save('y.npy', y)
    print(f"Rebuilt features: X={X.shape}, y={y.shape}")

print(f"Class balance (mean of y, 1=positive): {y.mean():.3f}")
print(f"Feature matrix dtype: {X.dtype}, label dtype: {y.dtype}")

# doc_vector and wv must exist on BOTH paths. On the load path above the rebuild
# branch never ran, so define them here if they are missing. Cells that turn raw
# text into a prediction (predict_sentiment, Lab 4) call doc_vector, and they must
# work whether the features were loaded or rebuilt.
if 'doc_vector' not in globals():
    import re
    import gensim.downloader as api
    wv = api.load('glove-wiki-gigaword-100')  # same 100-d vectors as B5

    def doc_vector(text):
        """Mean-pool the in-vocab word vectors of a text into one 100-d vector (B5 recipe)."""
        tokens = re.findall(r"[a-z']+", text.lower())
        vecs = [wv[t] for t in tokens if t in wv.key_to_index]  # OOV guard
        if not vecs:
            return np.zeros(INPUT_DIM, dtype=np.float32)
        return np.mean(vecs, axis=0).astype(np.float32)
    print("doc_vector defined on the load path (wv loaded for inference).")


## Section 1. Embeddings as Features, and the Bar to Beat

You now have `X`, an `(N, 100)` matrix where each row is one sentence turned into a vector by
averaging its word2vec vectors, and `y`, the sentiment label (0 = negative, 1 = positive).

This is the **embeddings as features** pattern. The pretrained word vectors are a frozen feature
extractor: we never train them. We only train a small classifier head that reads the 100-d
vector and predicts a label. That is cheap (no GPU bill per request), it works with very little
labeled data, and it is exactly what you do before deciding whether a full fine-tune is worth it.

Before we build a neural network, we need to know what "good" means. The honest way to justify a
neural network is to first fit the simplest reasonable model on the same features and treat its
score as the bar to beat. Here that model is **LogisticRegression**: a single linear layer with a
sigmoid, the same idea as one `nn.Linear` with no hidden layer. If our MLP cannot beat it, the
extra complexity is not earning its keep.

One detail that matters for both models: the 100 features live on different scales, and both
logistic regression and neural nets train better when features are standardized to zero mean and
unit variance. We fit the scaler on the TRAINING split only, then apply it to validation and test,
so no test information leaks into training.

**Figure: Embeddings as features - text becomes a frozen vector, only the small head trains.**

```mermaid
graph TD
    A[Raw sentence] --> B[doc_vector mean-pool word2vec]
    B --> C[100-d feature vector]
    C --> D[StandardScaler fit on train only]
    D --> E[Frozen features no gradient]
    E --> F[Small trainable head]
    F --> G[Sentiment label neg or pos]
```


In [ ]:
# Demo: split, standardize, and fit the LogisticRegression baseline.
# We split ONCE into train / val / test (70 / 15 / 15), stratified, reproducible with SEED.
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)
print(f"Train: {X_train.shape[0]}  Val: {X_val.shape[0]}  Test: {X_test.shape[0]}")

# Standardize: fit on TRAIN only, then apply the same transform to val and test (no leakage).
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

# LogisticRegression baseline on the standardized features.
baseline = LogisticRegression(max_iter=1000)
baseline.fit(X_train_s, y_train)
baseline_val_acc = accuracy_score(y_val, baseline.predict(X_val_s))
print(f"\nBASELINE (LogisticRegression) validation accuracy: {baseline_val_acc:.4f}")
print("This is the bar your MLP must beat.")

### Lab 1: Lock in the baseline (guided, ~10 min)

You just saw the baseline on the VALIDATION split. To report it fairly later, also compute the
baseline accuracy and macro-F1 on the held-out TEST split, using the already-fitted `scaler` and
`baseline` model from the demo.

Steps:

1. Transform `X_test` with the SAME fitted `scaler` (do not fit again).
2. Use the fitted `baseline` to predict labels for the transformed test features.
3. Compute accuracy and macro-F1 against `y_test`.
4. Store the accuracy in `baseline_test_acc`; the verification block checks it.

Reminder: the scaler was fit on train. Re-fitting on test would leak test statistics into your
numbers. Use the transform-only step.

In [ ]:
# Lab 1: compute the baseline on the TEST split using the already-fitted scaler and baseline.

# Step 1: apply the fitted scaler to the test features (transform only, do NOT fit here).
X_test_baseline = None  # YOUR CODE: transform X_test with the scaler fitted on train

# Step 2: predict test labels with the fitted baseline model.
baseline_test_preds = None  # YOUR CODE: use the baseline to predict on the transformed test features

# Step 3: score against y_test.
baseline_test_acc = None  # YOUR CODE: accuracy of baseline_test_preds vs y_test
baseline_test_f1  = None  # YOUR CODE: macro-F1 of baseline_test_preds vs y_test

# --- verification (provided) ---
assert X_test_baseline is not None and baseline_test_preds is not None, "Fill in steps 1 and 2."
assert baseline_test_acc is not None, "Compute baseline_test_acc."
assert X_test_baseline.shape == X_test.shape, "Use transform (not fit_transform); shape must match X_test."
assert 0.0 <= baseline_test_acc <= 1.0, "Accuracy must be a fraction in [0, 1]."
print(f"Baseline TEST accuracy: {baseline_test_acc:.4f} | macro-F1: {baseline_test_f1:.4f}")
print("Good. Now let's see if a neural net can beat this on the SAME features.")

In [ ]:
# Safety-net (provided): if Lab 1 above was skipped, supply working baseline test scores
# so the evaluation cell can still print the MLP-vs-baseline comparison.
if baseline_test_acc is None:
    print("Lab 1 was skipped; computing the baseline test scores with the reference recipe.")
    X_test_baseline = scaler.transform(X_test)
    baseline_test_preds = baseline.predict(X_test_baseline)
    baseline_test_acc = accuracy_score(y_test, baseline_test_preds)
    baseline_test_f1 = f1_score(y_test, baseline_test_preds, average='macro')
    print(f"Fallback baseline TEST accuracy: {baseline_test_acc:.4f} | macro-F1: {baseline_test_f1:.4f}")


## Section 2. The MLP: a Nonlinear Head on Frozen Features

Logistic regression draws ONE straight decision boundary in the 100-d feature space. That is all
a linear model can do. Many real boundaries are not straight, and that is exactly the gap a
neural network fills.

Our model adds one hidden layer with a nonlinear activation:

    input (100-d averaged word2vec)
      -> Linear(100, 128)     # learn 128 combinations of the input features
      -> ReLU                 # the nonlinearity: this is what beats a linear model
      -> Dropout(0.3)         # randomly zero some hidden units during training (regularization)
      -> Linear(128, 2)       # logits for [negative, positive]

The hidden layer plus ReLU lets the network bend the decision boundary: it composes the 100
inputs into 128 new features, and the output layer draws its line in that bent space. That is why
an MLP can beat logistic regression on the SAME inputs.

Two rules we carry from B6 and must not break:

- The model outputs RAW LOGITS. We do NOT put a softmax in `forward`. `nn.CrossEntropyLoss`
  applies log-softmax internally, and adding our own would double-apply it and hurt training.
- The labels are INTEGER class indices (0 or 1) with dtype `long`, never one-hot vectors.

Dropout is on only during training (`model.train()`) and off during evaluation (`model.eval()`).
Forgetting `model.eval()` at test time is a classic bug that makes results noisy.

**Figure: The SentimentMLP - Linear, ReLU, Dropout, Linear, emitting raw logits.**

```mermaid
graph TD
    A[Input 100-d averaged word2vec] --> B[Linear 100 to 128]
    B --> C[ReLU nonlinearity]
    C --> D[Dropout 0.3 train only]
    D --> E[Linear 128 to 2]
    E --> F[Raw logits neg pos]
    F --> G[CrossEntropyLoss applies softmax]
```


### Lab 2: Wrap the features in DataLoaders (guided, ~8 min)

A PyTorch training loop pulls mini-batches from a `DataLoader`. In B6 you built `DataLoader`s
from a `TensorDataset`. Do the same here with the standardized feature splits.

You need three things per split: float32 feature tensors, int64 (long) label tensors, and a
`TensorDataset` wrapping them. Then a `DataLoader` over each.

Steps:

1. Convert `X_train_s`, `X_val_s`, `X_test_s` to float tensors, and `y_train`, `y_val`, `y_test`
   to long tensors.
2. Build a `TensorDataset` for each split.
3. Build a `DataLoader` for each. Shuffle the TRAINING loader (use `BATCH_SIZE`); the val and
   test loaders do not need shuffling.

Why long labels? `CrossEntropyLoss` expects integer class indices, and PyTorch represents those
as the `long` dtype. Float labels raise an error here.

In [ ]:
# Lab 2: build TensorDatasets and DataLoaders from the standardized features.

# Step 1: tensors. Features must be float; labels must be long (integer class indices).
X_train_t = torch.tensor(X_train_s, dtype=torch.float32)
X_val_t   = torch.tensor(X_val_s,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test_s,  dtype=torch.float32)
y_train_t = None  # YOUR CODE: y_train as a torch long tensor
y_val_t   = None  # YOUR CODE: y_val as a torch long tensor
y_test_t  = None  # YOUR CODE: y_test as a torch long tensor

# Step 2: datasets (pair each feature tensor with its label tensor).
train_ds = None  # YOUR CODE: TensorDataset of train features and labels
val_ds   = None  # YOUR CODE: TensorDataset of val features and labels
test_ds  = None  # YOUR CODE: TensorDataset of test features and labels

# Step 3: loaders. Shuffle only the training loader.
train_loader = None  # YOUR CODE: DataLoader over train_ds, batch_size=BATCH_SIZE, shuffled
val_loader   = None  # YOUR CODE: DataLoader over val_ds, batch_size=BATCH_SIZE
test_loader  = None  # YOUR CODE: DataLoader over test_ds, batch_size=BATCH_SIZE

# --- verification (provided) ---
assert y_train_t is not None and y_train_t.dtype == torch.long, "Labels must be torch.long."
assert train_ds is not None and len(train_ds) == X_train_s.shape[0], "train_ds size mismatch."
xb, yb = next(iter(train_loader))
assert xb.shape[1] == INPUT_DIM, f"Each batch row must be {INPUT_DIM}-d."
assert yb.dtype == torch.long, "Batched labels must be long."
print(f"Loaders ready. One train batch: features {tuple(xb.shape)}, labels {tuple(yb.shape)}.")

In [ ]:
# Safety-net (provided): if Lab 2 above was skipped, build the DataLoaders here so the
# training loop and evaluation downstream still run.
if train_loader is None or val_loader is None or test_loader is None:
    print("Lab 2 was skipped; building TensorDatasets and DataLoaders with the reference recipe.")
    y_train_t = torch.tensor(y_train, dtype=torch.long)
    y_val_t   = torch.tensor(y_val,   dtype=torch.long)
    y_test_t  = torch.tensor(y_test,  dtype=torch.long)
    train_ds = TensorDataset(torch.tensor(X_train_s, dtype=torch.float32), y_train_t)
    val_ds   = TensorDataset(torch.tensor(X_val_s,   dtype=torch.float32), y_val_t)
    test_ds  = TensorDataset(torch.tensor(X_test_s,  dtype=torch.float32), y_test_t)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
    test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)
    print("Fallback DataLoaders ready.")


In [ ]:
# Lab (Tier 2): build the model class yourself. A plain dense MLP over the 100-d
# averaged-word2vec features. The architecture is in Section 2 above:
#   Linear(INPUT_DIM, HIDDEN_DIM) -> ReLU -> Dropout(DROPOUT) -> Linear(HIDDEN_DIM, N_CLASSES)
# Assemble it from nn building blocks. forward must return RAW LOGITS (no softmax).

class SentimentMLP(nn.Module):
    """MLP head on frozen averaged-word2vec features: Linear -> ReLU -> Dropout -> Linear."""

    def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM,
                 n_classes=N_CLASSES, dropout=DROPOUT):
        super().__init__()
        # YOUR CODE: define the layers (two Linear, one ReLU, one Dropout) as attributes.
        None

    def forward(self, x):
        # YOUR CODE: x [batch, input_dim] -> logits [batch, n_classes]. No softmax here.
        None

# Quick shape sanity check with a dummy batch (no training yet).
demo_model = SentimentMLP().to(device)
dummy = torch.randn(4, INPUT_DIM, device=device)
out = demo_model(dummy)

# --- verification (provided) ---
assert tuple(out.shape) == (4, N_CLASSES), f"Expected logits (4, {N_CLASSES}), got {tuple(out.shape)}."
assert any(isinstance(m, nn.ReLU) for m in demo_model.modules()), "Include a ReLU nonlinearity."
assert any(isinstance(m, nn.Dropout) for m in demo_model.modules()), "Include a Dropout layer."
print(f"Dummy forward: input {tuple(dummy.shape)} -> logits {tuple(out.shape)} (expect (4, 2)).")
print("Logits are raw scores, not probabilities. CrossEntropyLoss handles the softmax.")


In [ ]:
# Safety-net (provided): if the SentimentMLP lab above was skipped, define the class here
# so the helpers, instantiation, training, and save all still run.
if 'SentimentMLP' not in globals():
    print("The SentimentMLP lab was skipped; defining the reference class.")

    class SentimentMLP(nn.Module):
        """MLP head on frozen averaged-word2vec features: Linear -> ReLU -> Dropout -> Linear."""

        def __init__(self, input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM,
                     n_classes=N_CLASSES, dropout=DROPOUT):
            super().__init__()
            self.fc1 = nn.Linear(input_dim, hidden_dim)
            self.act = nn.ReLU()
            self.dropout = nn.Dropout(dropout)
            self.fc2 = nn.Linear(hidden_dim, n_classes)

        def forward(self, x):
            h = self.act(self.fc1(x))
            h = self.dropout(h)
            return self.fc2(h)

if 'demo_model' not in globals() or demo_model is None:
    demo_model = SentimentMLP().to(device)
    print("Fallback demo_model created for the parameter-count demo.")


In [ ]:
# Demo: parameter count and the train/eval helpers (the B6 training loop, packaged).
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable parameters: {count_parameters(demo_model):,}")
print("Tiny model: every parameter is in the two Linear layers (the word vectors are frozen).")

def train_one_epoch(model, loader, loss_fn, optimizer, device):
    """One training pass; returns average loss."""
    model.train()  # dropout ON
    total = 0.0
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / len(loader)

def evaluate(model, loader, loss_fn, device):
    """Evaluate; returns (average loss, accuracy)."""
    model.eval()  # dropout OFF
    total = 0.0
    preds_all, labels_all = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            logits = model(xb)
            total += loss_fn(logits, yb).item()
            preds_all.extend(torch.argmax(logits, dim=1).cpu().numpy())
            labels_all.extend(yb.cpu().numpy())
    return total / len(loader), accuracy_score(labels_all, preds_all)

print("Helpers ready: train_one_epoch, evaluate.")

In [ ]:
# Lab 3a: create the real model, the loss function, and the optimizer.
torch.manual_seed(SEED)  # re-seed so weight init is reproducible

model = None      # YOUR CODE: a SentimentMLP, moved to device
loss_fn = None    # YOUR CODE: the loss for multi-class logits + integer labels
optimizer = None  # YOUR CODE: Adam over the model parameters with learning rate LR

# --- verification (provided) ---
assert isinstance(model, SentimentMLP), "model must be a SentimentMLP."
assert next(model.parameters()).device.type == device.type, "model must be on `device`."
assert loss_fn is not None and optimizer is not None, "Set loss_fn and optimizer."
_xb, _yb = next(iter(train_loader))
_loss = loss_fn(model(_xb.to(device)), _yb.to(device))
print(f"Setup OK. One untrained batch loss: {_loss.item():.4f} (around ln(2)=0.69 before training).")

In [ ]:
# Safety-net (provided): if Lab 3a above was skipped, build the model, loss, and optimizer
# here so the training loop has something to train.
if model is None or loss_fn is None or optimizer is None:
    print("Lab 3a was skipped; creating model, loss, and optimizer with the reference recipe.")
    torch.manual_seed(SEED)
    model = SentimentMLP().to(device)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    print("Fallback model, loss_fn, and optimizer ready.")


**Figure: The training loop with early stopping - save the best checkpoint, then test once.**

```mermaid
graph TD
    A[Train one epoch on train loader] --> B[Evaluate on val loader]
    B --> C{Val accuracy improved?}
    C -->|yes| D[Save best checkpoint reset patience]
    C -->|no| E[Increment patience counter]
    E --> F{Patience exceeded?}
    F -->|no| A
    F -->|yes| G[Early stop]
    D --> A
    G --> H[Reload best weights then test once]
```


In [ ]:
# Lab (Tier 2): the training loop with early stopping. The scaffolding is given; you wire
# the per-epoch work and the early-stopping decision. See the diagram above for the flow.
history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0
patience_counter = 0
best_path = 'best_b7_mlp.pt'

for epoch in range(EPOCHS_MAX):
    # YOUR CODE: one training pass over train_loader; keep the average loss in train_loss.
    train_loss = None
    # YOUR CODE: a validation pass over val_loader; capture (val_loss, val_acc).
    val_loss, val_acc = None, None

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    print(f"Epoch {epoch+1:02d} | train {train_loss:.4f} | val {val_loss:.4f} | val acc {val_acc:.4f}")

    # YOUR CODE: if this epoch's val_acc beat best_val_acc, update best_val_acc, reset
    # patience_counter to 0, and torch.save(model.state_dict(), best_path). Otherwise
    # increment patience_counter and break out of the loop once it reaches PATIENCE.
    None

print(f"\nBest validation accuracy: {best_val_acc:.4f}")
print(f"Baseline to beat (val): {baseline_val_acc:.4f}")


In [ ]:
# Safety-net (provided): if the training loop above was skipped (history never populated
# or no checkpoint written), run the reference training loop now so the model is actually
# trained before we plot curves, reload the best weights, and test.
import os as _os
if not history['val_acc'] or not _os.path.exists(best_path):
    print("Training loop was skipped; running the reference training loop with early stopping.")
    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    patience_counter = 0
    for epoch in range(EPOCHS_MAX):
        train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer, device)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_counter += 1
            if patience_counter >= PATIENCE:
                break
    print(f"Fallback training done. Best validation accuracy: {best_val_acc:.4f}")


In [ ]:
# Demo: plot the learning curves, then reload the best checkpoint for a fair test.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.plot(history['train_loss'], label='train')
ax1.plot(history['val_loss'], label='val')
ax1.set_title('Loss'); ax1.set_xlabel('epoch'); ax1.legend(); ax1.grid(alpha=0.3)
ax2.plot(history['val_acc'], label='val acc', color='green')
ax2.axhline(baseline_val_acc, color='red', linestyle='--', label='LogReg baseline')
ax2.set_title('Validation accuracy vs baseline'); ax2.set_xlabel('epoch')
ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Reload the best weights (the last epoch is not always the best). weights_only=True is the
# safe way to load a state_dict.
model.load_state_dict(torch.load(best_path, weights_only=True))
print("Reloaded best checkpoint. If val accuracy crossed the red line, the MLP beat the baseline.")

## Section 3. Did We Beat the Baseline?

The test set is the final exam: we touch it once, after all training and model selection. We
report accuracy (SST-2 is roughly balanced, so accuracy is meaningful) and macro-F1 (the average
of the per-class F1 scores), and we put the MLP and the baseline side by side.

In [ ]:
# Test the reloaded best model and compare to the LogisticRegression baseline.
test_loss, test_acc = evaluate(model, test_loader, loss_fn, device)

# Gather predictions for the report and confusion matrix.
model.eval()
mlp_preds, mlp_labels = [], []
with torch.no_grad():
    for xb, yb in test_loader:
        logits = model(xb.to(device))
        mlp_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
        mlp_labels.extend(yb.numpy())

mlp_test_f1 = f1_score(mlp_labels, mlp_preds, average='macro')

print("=== Test results ===")
print(f"MLP        accuracy: {test_acc:.4f} | macro-F1: {mlp_test_f1:.4f}")
print(f"Baseline   accuracy: {baseline_test_acc:.4f} | macro-F1: {baseline_test_f1:.4f}")
print(f"Lift (MLP - baseline): {test_acc - baseline_test_acc:+.4f}")
print()
print(classification_report(mlp_labels, mlp_preds, target_names=['negative', 'positive']))

In [ ]:
# Demo: confusion matrix on the test set.
cm = confusion_matrix(mlp_labels, mlp_preds)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['neg', 'pos'], yticklabels=['neg', 'pos'])
plt.xlabel('predicted'); plt.ylabel('true'); plt.title('Confusion Matrix (test)')
plt.tight_layout(); plt.show()

# Which direction does the model err more? That hints at what the averaged features miss.
fn = cm[1, 0]  # true positive predicted negative
fp = cm[0, 1]  # true negative predicted positive
print(f"Positives called negative: {fn} | Negatives called positive: {fp}")

In [ ]:
# Demo: a tiny inference function. Text -> doc_vector -> scale -> model -> label.
# This is the embeddings-as-features pipeline end to end, the shape a deployed service has.
import torch.nn.functional as Fnn

def predict_sentiment(text):
    """Return (label_str, confidence) for a raw sentence using the trained MLP."""
    vec = doc_vector(text).reshape(1, -1)          # 100-d averaged word2vec (B5 recipe)
    vec = scaler.transform(vec)                    # SAME scaler fit on train
    t = torch.tensor(vec, dtype=torch.float32, device=device)
    model.eval()
    with torch.no_grad():
        probs = Fnn.softmax(model(t), dim=1)[0]    # softmax here, for a readable confidence
    label = 'positive' if probs[1] > probs[0] else 'negative'
    return label, float(probs.max())

for s in ["a delightful, warm-hearted film", "a boring, lifeless mess"]:
    label, conf = predict_sentiment(s)
    print(f"{label:8s} ({conf:.2f})  <-  {s}")

### Lab 4: Break the model on purpose (guided, ~10 min)

Averaging word vectors throws away word ORDER. "not good" and "good not" average to the same
vector, and "not good" shares almost every word with "good". So the model should struggle with
negation and word order.

Your task: write three short sentences where word order or negation flips the true sentiment, run
them through `predict_sentiment`, and find at least one the model gets WRONG. Then write one
sentence in the `note` string explaining why averaged features cannot capture it.

You are hunting for a failure, not avoiding one. A clean negation like "this is not good at all"
is a great candidate.

**Figure: The averaging ceiling - why not good and good collapse to nearly the same vector.**

```mermaid
graph TD
    A[good] --> C[Mean-pool word vectors]
    B[not good] --> C
    C --> D[Nearly identical 100-d vectors]
    D --> E[Model cannot tell them apart]
    E --> F[Negation and order are lost]
    F --> G[The averaging ceiling]
    G --> H[Attention in Part C fixes this]
```


In [ ]:
# Lab 4: adversarial sentences that stress word order / negation.
adversarial = [
    None,  # YOUR CODE: a sentence whose sentiment depends on a negation word
    None,  # YOUR CODE: a sentence whose sentiment depends on word order
    None,  # YOUR CODE: your own tricky case
]
note = None  # YOUR CODE: one sentence on why averaging loses this information

# --- verification (provided) ---
assert all(isinstance(s, str) and s.strip() for s in adversarial), "Provide three real sentences."
assert isinstance(note, str) and len(note) > 20, "Write a one-sentence explanation in `note`."
for s in adversarial:
    label, conf = predict_sentiment(s)
    print(f"{label:8s} ({conf:.2f})  <-  {s}")
print(f"\nYour explanation: {note}")

## Section 4. Save It Like You Mean to Deploy It

To reuse this classifier later you need TWO artifacts, not one:

1. The model weights, saved as a `state_dict` (portable, the recommended way; saving the whole
   pickled model object is brittle across versions).
2. The fitted `StandardScaler`. Inference must apply the exact same scaling that training used,
   so the scaler is part of the model, not an afterthought.

In production you would also precompute and cache the document vectors for any static corpus, so
serving a request is just `doc_vector -> scale -> tiny MLP`, which is fast and cheap precisely
because the encoder is frozen.

In [ ]:
# Save both artifacts.
import pickle
torch.save(model.state_dict(), 'b7_sentiment_mlp.pt')
with open('b7_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Saved b7_sentiment_mlp.pt (weights) and b7_scaler.pkl (scaler).")
print("To load: rebuild SentimentMLP, load_state_dict(..., weights_only=True), call model.eval().")

## Wrap-up and Homework

### What you did

- Turned text into 100-d averaged word2vec features (the B5 recipe), standardized them, and fit
  a LogisticRegression baseline: the bar to beat.
- Built and trained an MLP head with the B6 machinery and BEAT that baseline on the same features.
- Saw, in a confusion matrix and a negation example, exactly where averaging hits its ceiling.
- Saved the model and scaler the way a deployable service needs them.

The big idea: **embeddings as features**. A frozen pretrained encoder plus a small trained head
is the cheapest useful classifier you can ship, and it is the exact pattern Part C upgrades by
making the encoder trainable.

### Homework Extension (async, the real lift): swap in contextual sentence embeddings

The MLP's ceiling is the REPRESENTATION, not the model. Averaged static word vectors cannot see
order or context. Contextual sentence embeddings can. Prove it:

1. Encode the SAME SST-2 sentences with the B5 sentence encoder
   `embedder = SentenceTransformer('all-MiniLM-L6-v2')` to get a 384-d feature matrix
   `X_sbert` (shape `(N, 384)`). The labels `y` are unchanged.
2. Re-run the EXACT pipeline: stratified split, `StandardScaler`, the same `SentimentMLP` but
   with `input_dim=384`, the same training loop and early stopping.
3. Compare test accuracy. On SST, averaged GloVe lands around 80 percent; an SBERT-style sentence
   embedding lands closer to 89 percent. Report YOUR lift and write two sentences on why the
   contextual model wins.

This is the bridge to Part C: SBERT is a frozen contextual encoder; C9 goes one step further and
FINE-TUNES the encoder (DistilBERT) on SST-2, which beats even SBERT, and that fine-tuned model is
what the Gradio chatbot loads.

### Stretch options (pick one, in-notebook)

- **More capacity, more regularization**: add a second hidden layer and try `Adam(..., weight_decay=1e-4)`.
  Does accuracy improve, or does the averaged-feature ceiling cap it regardless? (Usually the
  representation, not the model size, is the limit here.)
- **Freeze vs scale ablation**: retrain WITHOUT the StandardScaler and compare convergence and
  final accuracy. Quantify how much scaling bought you.
- **Hyperparameter sweep**: vary `HIDDEN_DIM` over {32, 128, 256} and plot val accuracy. Report
  the point of diminishing returns.

### Production take-homes

- Precompute and cache embeddings for static corpora; serve only the small head online.
- Use frozen features for low-data / quick-prototype settings; fine-tune when you have the data
  and compute and need task-specific accuracy.
- A softmax confidence is often miscalibrated. For a deployed classifier, calibrate the scores
  and tune the decision threshold, and monitor the confidence distribution for drift.

### Bridge to B8 (Capstone B)

You own the full pipeline now: pretrained features in, trained classifier out, baseline beaten,
model saved. Capstone B runs it end to end on a fresh dataset. Then Part C swaps the frozen
encoder for a fine-tuned transformer.

### Resources

- gensim KeyedVectors: https://radimrehurek.com/gensim/models/keyedvectors.html
- PyTorch CrossEntropyLoss: https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html
- Saving for inference: https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html
- Sentence-BERT (Reimers and Gurevych, 2019): https://arxiv.org/abs/1908.10084

**Figure: Frozen features to fine-tune - the bridge from B7 to the SBERT homework and C9.**

```mermaid
flowchart LR
    A[B7 frozen word2vec plus MLP head] --> B[Beats LogReg baseline around 80 pct]
    B --> C[Homework frozen SBERT 384-d plus same head]
    C --> D[Lift to around 89 pct]
    D --> E[C9 fine-tune DistilBERT encoder itself]
    E --> F[Beats SBERT loaded into Gradio chatbot]
```
